In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the data
df = pd.read_csv('../data/austin_housing_cleaned.csv')

# Filter for properties with school ratings
df_schools = df[['avgSchoolRating', 'latestPrice', 'livingAreaSqFt', 'numOfBedrooms', 'numOfBathrooms']].dropna()

print(f"Dataset size: {len(df_schools)} properties")
print(f"\nSchool Rating Range: {df_schools['avgSchoolRating'].min():.1f} to {df_schools['avgSchoolRating'].max():.1f}")
print(f"Average School Rating: {df_schools['avgSchoolRating'].mean():.2f}")
print(f"\nPrice Range: ${df_schools['latestPrice'].min():,.0f} to ${df_schools['latestPrice'].max():,.0f}")
print(f"Average Price: ${df_schools['latestPrice'].mean():,.0f}")

In [ ]:
# Simple regression: School Rating vs Price (controlling for size)
X = df_schools[['avgSchoolRating', 'livingAreaSqFt', 'numOfBedrooms', 'numOfBathrooms']]
y = df_schools['latestPrice']

model = LinearRegression()
model.fit(X, y)

# Get the coefficient for school rating
school_coef = model.coef_[0]

print(f"\n{'='*70}")
print(f"SCHOOL QUALITY DOLLAR IMPACT")
print(f"{'='*70}")
print(f"\nFor each 1-point increase in school rating:")
print(f"  Home value increases by: ${school_coef:,.0f}")
print(f"\nFor a 'Poor' school (rating 3) vs 'Good' school (rating 7):")
print(f"  Difference: 4 rating points")
print(f"  Price impact: ${school_coef * 4:,.0f}")
print(f"\nFor a 'Poor' school (rating 3) vs 'Excellent' school (rating 9):")
print(f"  Difference: 6 rating points")
print(f"  Price impact: ${school_coef * 6:,.0f}")
print(f"\n{'='*70}")

In [ ]:
# Show school rating categories and their average prices
df_schools['school_category'] = pd.cut(
    df_schools['avgSchoolRating'],
    bins=[0, 4, 6, 8, 10],
    labels=['Poor (0-4)', 'Fair (4-6)', 'Good (6-8)', 'Excellent (8-10)']
)

category_stats = df_schools.groupby('school_category').agg({
    'latestPrice': ['mean', 'median', 'count'],
    'avgSchoolRating': 'mean'
}).round(0)

print("\nAverage Prices by School Quality Category:")
print("="*70)
for cat in category_stats.index:
    avg_price = category_stats.loc[cat, ('latestPrice', 'mean')]
    median_price = category_stats.loc[cat, ('latestPrice', 'median')]
    count = category_stats.loc[cat, ('latestPrice', 'count')]
    avg_rating = category_stats.loc[cat, ('avgSchoolRating', 'mean')]
    print(f"\n{cat}:")
    print(f"  Avg Rating: {avg_rating:.1f}")
    print(f"  Avg Price: ${avg_price:,.0f}")
    print(f"  Median Price: ${median_price:,.0f}")
    print(f"  Count: {count:.0f} properties")

In [ ]:
# Calculate the price difference between poor and good schools
poor_schools = df_schools[df_schools['avgSchoolRating'] <= 4]
good_schools = df_schools[df_schools['avgSchoolRating'] >= 7]

if len(poor_schools) > 0 and len(good_schools) > 0:
    poor_avg = poor_schools['latestPrice'].mean()
    good_avg = good_schools['latestPrice'].mean()
    difference = good_avg - poor_avg
    percent_diff = (difference / poor_avg) * 100
    
    print(f"\n{'='*70}")
    print(f"REAL-WORLD COMPARISON")
    print(f"{'='*70}")
    print(f"\nPoor Schools (rating ≤ 4):")
    print(f"  Average Price: ${poor_avg:,.0f}")
    print(f"  Number of Properties: {len(poor_schools)}")
    print(f"\nGood Schools (rating ≥ 7):")
    print(f"  Average Price: ${good_avg:,.0f}")
    print(f"  Number of Properties: {len(good_schools)}")
    print(f"\nDIFFERENCE:")
    print(f"  Dollar Amount: ${difference:,.0f}")
    print(f"  Percentage: {percent_diff:.1f}% more expensive")
    print(f"\n{'='*70}")
else:
    print("Insufficient data for poor/good school comparison")

In [ ]:
# Summary
print(f"\n{'='*70}")
print(f"SUMMARY: WHAT IS A POOR QUALITY SCHOOL WORTH?")
print(f"{'='*70}")
print(f"\nBased on {len(df_schools):,} Austin properties:")
print(f"\n1. Per Rating Point Impact: ${abs(school_coef):,.0f}")
print(f"   Each 1-point decrease in school rating reduces home value by ~${abs(school_coef):,.0f}")
print(f"\n2. Poor School Penalty (rating 3 vs 7):")
print(f"   A home near poor schools is worth ${abs(school_coef * 4):,.0f} LESS")
print(f"   than a comparable home near good schools")
print(f"\n3. Model R² Score: {model.score(X, y):.4f}")
print(f"   (explains {model.score(X, y)*100:.1f}% of price variation)")
print(f"\n{'='*70}")